# v3.6 New Model: Florence-2-Large Grounded Captioning

Florence-2 from Microsoft provides grounded image captioning (MIT license).
Uses `microsoft/Florence-2-large` via HuggingFace Transformers.

In [ ]:
from PIL import Image
img = Image.new('RGB', (512, 512), color=(180, 200, 220))
print('Image:', img.size)

In [ ]:
try:
    from transformers import AutoProcessor, AutoModelForCausalLM
    import torch, time, json
    from pathlib import Path

    model_id = 'microsoft/Florence-2-large'
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, torch_dtype=torch.float32)

    prompt = '<MORE_DETAILED_CAPTION>'
    inputs = processor(text=prompt, images=img, return_tensors='pt')

    t0 = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128, num_beams=3)
    latency_ms = (time.perf_counter() - t0) * 1000

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    result = processor.post_process_generation(generated_text, task=prompt, image_size=img.size)

    print(f'Florence-2-large caption ({latency_ms:.0f} ms):', result)

    artifact = {'model_id': 'florence-2-large', 'task': 'grounded_caption', 'caption': str(result), 'latency_ms': latency_ms}
    Path('/tmp/florence2_large_caption.json').write_text(json.dumps(artifact, indent=2))
    print('Artifact saved.')
except Exception as e:
    print(f'Skipped (missing deps): {e}')